# Static finance metric → mentions as % of STV threshold, by district (2024)

## Question

**For each Portland City Council district, how is a selected finance metric associated with ballot mentions relative to the district's STV threshold?**

This notebook is a direct adaptation of the team's static Notebook 13b.

The plotting and regression logic is intentionally kept the same:

1. fit `statsmodels.OLS`;
2. draw the line of best fit;
3. draw the 95% confidence interval;
4. report Pearson r and R²;
5. label candidates;
6. highlight candidates with similar finance values but very different electoral support.

The difference is that the notebook now loops through **Districts 1–4** and automatically saves one figure per district.

The Y variable is:

`mentions_pct_threshold = 100 × mentions / stv_threshold`

So **100% means that a candidate's total mentions equal the STV election threshold in that district**.

> Descriptive association only. These regressions do not establish causation.


## 1. Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from IPython.display import display


# ---------------------------------------------------------------
# Find the repository root
# ---------------------------------------------------------------

cwd = Path.cwd().resolve()

for folder in [cwd, *cwd.parents]:
    if (folder / "pyproject.toml").exists():
        ROOT = folder
        break
else:
    raise FileNotFoundError(
        "Could not find the repository root."
    )


if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT),
    )


from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)


YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


## 2. Load ballot support + fundraising + spending profiles

This keeps the same three canonical inputs used by the original static Notebook 13b.


In [ ]:
SUPPORT_PATH = (
    PROCESSED
    / "ballot_support"
    / str(YEAR)
    / "candidate_ballot_support_2024.csv"
)

FUNDRAISING_PATH = (
    fundraising_processed_dir(
        YEAR,
        CONTEST,
    )
    / "openelections_candidate_fundraising_profiles_wide.csv"
)

SPENDING_PATH = (
    spending_processed_dir(
        YEAR,
        CONTEST,
    )
    / "orestar_candidate_spending_profiles_wide.csv"
)


support = pd.read_csv(
    SUPPORT_PATH,
    low_memory=False,
)

fundraising = pd.read_csv(
    FUNDRAISING_PATH,
    low_memory=False,
)

spending = pd.read_csv(
    SPENDING_PATH,
    low_memory=False,
)


# Keep only spending rows matched to an official candidate.
spending = spending[
    spending["candidate_key"].notna()
].copy()


if spending["candidate_key"].duplicated().any():
    raise ValueError(
        "Duplicate candidate_key values found in spending profiles."
    )


print("Ballot-support candidates:", len(support))
print("Fundraising profiles:", len(fundraising))
print("Linked spending profiles:", len(spending))


### Keep only the columns we need before merging

This follows the original static notebook so the selected finance metric can still be changed from one control cell.


In [ ]:
# ---------------------------------------------------------------
# Fundraising columns
# ---------------------------------------------------------------

fundraising_columns = [
    "year",
    "district",
    "candidate_key",
    "total_amount",
    "total_contribution_count",
    "average_contribution",
    "median_contribution",
]

for contribution_bin in [
    "micro",
    "small",
    "medium",
    "large",
    "mega",
]:
    fundraising_columns.extend(
        [
            f"amount_{contribution_bin}",
            f"amount_share_{contribution_bin}",
            f"contribution_count_{contribution_bin}",
            f"contribution_share_{contribution_bin}",
        ]
    )


# ---------------------------------------------------------------
# Spending columns
# ---------------------------------------------------------------

spending_columns = [
    "year",
    "district",
    "candidate_key",
    "total_spending",
    "expenditure_count",
    "average_expenditure",
    "median_expenditure",
]

for spending_bin in [
    "micro",
    "small",
    "medium",
    "large",
    "mega",
]:
    spending_columns.extend(
        [
            f"spending_amount_{spending_bin}",
            f"spending_amount_share_{spending_bin}",
            f"spending_count_{spending_bin}",
            f"spending_count_share_{spending_bin}",
        ]
    )


# Keep only columns that actually exist.
fundraising_columns = [
    column
    for column in fundraising_columns
    if column in fundraising.columns
]

spending_columns = [
    column
    for column in spending_columns
    if column in spending.columns
]


analysis = (
    support
    .merge(
        fundraising[
            fundraising_columns
        ],
        on=[
            "year",
            "district",
            "candidate_key",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        spending[
            spending_columns
        ],
        on=[
            "year",
            "district",
            "candidate_key",
        ],
        how="left",
        validate="one_to_one",
    )
)


# Friendly alias used in the original static notebook.
analysis["total_fundraising"] = (
    analysis["total_amount"]
)


print("Analysis rows:", len(analysis))
print("Analysis columns:", len(analysis.columns))


## 3. Build mentions as % of the district threshold

This is derived inside the notebook; no new processed CSV is needed.


In [ ]:
analysis["mentions_pct_threshold"] = (
    100
    * analysis["mentions"]
    / analysis["stv_threshold"]
)


display(
    analysis[
        [
            "district",
            "canonical_candidate",
            "mentions",
            "stv_threshold",
            "mentions_pct_threshold",
        ]
    ]
    .sort_values(
        [
            "district",
            "mentions_pct_threshold",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        {
            "mentions_pct_threshold": 1,
        }
    )
)


## 4. Finance variables we can test

In [ ]:
METRICS = {}


# ---------------------------------------------------------------
# A. Candidate-level totals
# ---------------------------------------------------------------

METRICS["fundraising_total"] = {
    "column": "total_fundraising",
    "label": "Total fundraising ($)",
    "family": "Fundraising total",
}

METRICS["fundraising_count"] = {
    "column": "total_contribution_count",
    "label": "Contribution records",
    "family": "Fundraising total",
}

METRICS["fundraising_mean"] = {
    "column": "average_contribution",
    "label": "Mean contribution ($)",
    "family": "Fundraising total",
}

METRICS["fundraising_median"] = {
    "column": "median_contribution",
    "label": "Median contribution ($)",
    "family": "Fundraising total",
}


METRICS["spending_total"] = {
    "column": "total_spending",
    "label": "Total spending ($)",
    "family": "Spending total",
}

METRICS["spending_count"] = {
    "column": "expenditure_count",
    "label": "Expenditure records",
    "family": "Spending total",
}

METRICS["spending_mean"] = {
    "column": "average_expenditure",
    "label": "Mean expenditure ($)",
    "family": "Spending total",
}

METRICS["spending_median"] = {
    "column": "median_expenditure",
    "label": "Median expenditure ($)",
    "family": "Spending total",
}


# ---------------------------------------------------------------
# B. Fundraising bins
# ---------------------------------------------------------------

for contribution_bin in [
    "micro",
    "small",
    "medium",
    "large",
    "mega",
]:
    pretty_bin = contribution_bin.title()

    METRICS[
        f"fundraising_amount_{contribution_bin}"
    ] = {
        "column": f"amount_{contribution_bin}",
        "label": f"{pretty_bin} contribution dollars ($)",
        "family": "Fundraising bin amount",
    }

    METRICS[
        f"fundraising_amount_share_{contribution_bin}"
    ] = {
        "column": f"amount_share_{contribution_bin}",
        "label": f"{pretty_bin} share of fundraising dollars",
        "family": "Fundraising bin share",
    }

    METRICS[
        f"fundraising_count_{contribution_bin}"
    ] = {
        "column": f"contribution_count_{contribution_bin}",
        "label": f"{pretty_bin} contribution records",
        "family": "Fundraising bin count",
    }

    METRICS[
        f"fundraising_count_share_{contribution_bin}"
    ] = {
        "column": f"contribution_share_{contribution_bin}",
        "label": f"{pretty_bin} share of contribution records",
        "family": "Fundraising bin count share",
    }


# ---------------------------------------------------------------
# C. Spending bins
# ---------------------------------------------------------------

for spending_bin in [
    "micro",
    "small",
    "medium",
    "large",
    "mega",
]:
    pretty_bin = spending_bin.title()

    METRICS[
        f"spending_amount_{spending_bin}"
    ] = {
        "column": f"spending_amount_{spending_bin}",
        "label": f"{pretty_bin} spending amount ($)",
        "family": "Spending bin amount",
    }

    METRICS[
        f"spending_amount_share_{spending_bin}"
    ] = {
        "column": f"spending_amount_share_{spending_bin}",
        "label": f"{pretty_bin} share of spending dollars",
        "family": "Spending bin share",
    }

    METRICS[
        f"spending_count_{spending_bin}"
    ] = {
        "column": f"spending_count_{spending_bin}",
        "label": f"{pretty_bin} expenditure records",
        "family": "Spending bin count",
    }

    METRICS[
        f"spending_count_share_{spending_bin}"
    ] = {
        "column": f"spending_count_share_{spending_bin}",
        "label": f"{pretty_bin} share of expenditure records",
        "family": "Spending bin count share",
    }


# Remove options whose source column is not present.
METRICS = {
    key: value
    for key, value in METRICS.items()
    if value["column"] in analysis.columns
}


metric_options = pd.DataFrame(
    [
        {
            "option": key,
            "family": value["family"],
            "variable": value["label"],
            "column": value["column"],
        }
        for key, value in METRICS.items()
    ]
)


display(
    metric_options
    .sort_values(
        [
            "family",
            "option",
        ]
    )
    .reset_index(
        drop=True
    )
)


# 5. CHANGE ONLY THIS CELL

Choose the finance metric once. The notebook will then run the same graph for Districts 1–4.


In [ ]:
# ===============================================================
# CHANGE THIS VALUE
# ===============================================================

X_VARIABLE = "fundraising_total"


# ---------------------------------------------------------------
# Usually leave these settings alone
# ---------------------------------------------------------------

DISTRICTS = [1, 2, 3, 4]

Y_VARIABLE = "mentions_pct_threshold"

SHOW_CANDIDATE_NAMES = True

# Candidates are considered similar on X when their selected
# finance measure differs by 10% or less.
SIMILAR_X_TOLERANCE = 0.10

# Highlight a pair only if Y differs by at least 50%.
LARGE_Y_GAP = 0.50

SAVE_FIGURES = True


## 6. OLS function

This is the same static Matplotlib / statsmodels OLS function used in Notebook 13b.


In [ ]:
def OLSlinearRegression(
    x_data,
    y_data,
    x_label,
    y_label,
    title,
    save_name="",
    colors=None,
    names=None,
):
    # -----------------------------------------------------------
    # Convert to numpy arrays
    # -----------------------------------------------------------

    x = np.asarray(
        x_data,
        dtype=np.float64,
    )

    y = np.asarray(
        y_data,
        dtype=np.float64,
    )


    # -----------------------------------------------------------
    # Fit regression model
    # -----------------------------------------------------------

    X = sm.add_constant(
        x
    )

    model = sm.OLS(
        y,
        X,
    ).fit()


    # -----------------------------------------------------------
    # Create x values for the line of best fit
    # -----------------------------------------------------------

    x_min = x.min()
    x_max = x.max()

    x_padding = (
        (x_max - x_min) * 0.05
        if x_max > x_min
        else 1
    )

    x_plot = np.linspace(
        x_min,
        x_max,
        300,
    )

    X_plot = sm.add_constant(
        x_plot
    )


    # -----------------------------------------------------------
    # Predictions + 95% confidence interval
    # -----------------------------------------------------------

    prediction = model.get_prediction(
        X_plot
    )

    prediction_summary = (
        prediction
        .summary_frame(
            alpha=0.05
        )
    )

    y_pred = prediction_summary[
        "mean"
    ].to_numpy(
        dtype=float
    )

    ci_lower = prediction_summary[
        "mean_ci_lower"
    ].to_numpy(
        dtype=float
    )

    ci_upper = prediction_summary[
        "mean_ci_upper"
    ].to_numpy(
        dtype=float
    )


    # -----------------------------------------------------------
    # Statistics
    # -----------------------------------------------------------

    r_squared = model.rsquared

    pearson_r = np.corrcoef(
        x,
        y,
    )[0, 1]


    print(
        f"n = {len(x)}"
    )

    print(
        f"Pearson r = {pearson_r:.3f}"
    )

    print(
        f"R² = {r_squared:.3f}"
    )

    print(
        f"Slope p-value = {model.pvalues[1]:.4g}"
    )


    # -----------------------------------------------------------
    # Make plot
    # -----------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(11, 8)
    )


    if colors is None:
        colors = [
            "C0"
        ] * len(x)


    for i in range(
        len(x)
    ):
        ax.scatter(
            x[i],
            y[i],
            color=colors[i],
            s=65,
            alpha=0.85,
            zorder=3,
        )


    # Regression line
    ax.plot(
        x_plot,
        y_pred,
        color="red",
        label="OLS fit",
        zorder=2,
    )


    # Confidence interval
    ax.fill_between(
        x_plot,
        ci_lower,
        ci_upper,
        color="red",
        alpha=0.15,
        label="95% CI",
        zorder=1,
    )


    # -----------------------------------------------------------
    # Candidate labels
    # -----------------------------------------------------------

    if (
        names is not None
        and SHOW_CANDIDATE_NAMES
    ):
        # Alternate label positions to reduce overlap.
        offsets = [
            (7, 8),
            (7, -12),
            (-7, 8),
            (-7, -12),
            (10, 0),
            (-10, 0),
        ]

        for i, name in enumerate(
            names
        ):
            x_offset, y_offset = (
                offsets[
                    i % len(offsets)
                ]
            )

            ax.annotate(
                str(name),
                (
                    x[i],
                    y[i],
                ),
                xytext=(
                    x_offset,
                    y_offset,
                ),
                textcoords="offset points",
                fontsize=8.5,
                ha=(
                    "left"
                    if x_offset >= 0
                    else "right"
                ),
                va="center",
            )


    # -----------------------------------------------------------
    # Labels and title
    # -----------------------------------------------------------

    ax.set_xlabel(
        x_label
    )

    ax.set_ylabel(
        y_label
    )

    ax.set_xlim(
        x_min - x_padding,
        x_max + x_padding,
    )

    ax.set_title(
        title
        + "\n"
        + (
            f"Pearson r = {pearson_r:.2f} | "
            f"R² = {r_squared:.2f} | "
            f"n = {len(x)}"
        )
    )

    ax.legend(
        frameon=False
    )

    plt.tight_layout()


    if save_name:
        plt.savefig(
            save_name,
            dpi=300,
            bbox_inches="tight",
        )

        print(
            "SAVED:",
            save_name,
        )


    plt.show()


    return model


## 7. Comparable candidate helpers

These preserve the same idea as Notebook 13b: candidates with similar X but a large gap in Y are colored together.


In [ ]:
def relative_gap(value_a, value_b):
    larger_value = max(
        abs(value_a),
        abs(value_b),
    )

    if larger_value == 0:
        return 0.0

    return (
        abs(
            value_a
            - value_b
        )
        / larger_value
    )


def find_interesting_pairs(
    data,
    x_column,
    y_column,
    x_tolerance=0.10,
    y_gap_threshold=0.50,
):
    rows = []

    data = data.reset_index(
        drop=True
    )


    for i in range(
        len(data)
    ):
        for j in range(
            i + 1,
            len(data),
        ):
            candidate_a = data.iloc[i]
            candidate_b = data.iloc[j]


            x_gap = relative_gap(
                candidate_a[x_column],
                candidate_b[x_column],
            )

            y_gap = relative_gap(
                candidate_a[y_column],
                candidate_b[y_column],
            )


            if (
                x_gap <= x_tolerance
                and y_gap >= y_gap_threshold
            ):
                rows.append(
                    {
                        "candidate_a": candidate_a[
                            "canonical_candidate"
                        ],
                        "candidate_b": candidate_b[
                            "canonical_candidate"
                        ],
                        "x_a": candidate_a[
                            x_column
                        ],
                        "x_b": candidate_b[
                            x_column
                        ],
                        "y_a": candidate_a[
                            y_column
                        ],
                        "y_b": candidate_b[
                            y_column
                        ],
                        "x_gap_pct": x_gap * 100,
                        "y_gap_pct": y_gap * 100,
                    }
                )


    return pd.DataFrame(
        rows
    )


def assign_case_groups(
    data,
    pairs,
):
    data = data.copy()

    candidate_names = (
        data[
            "canonical_candidate"
        ]
        .astype(str)
        .tolist()
    )


    connections = {
        name: set()
        for name in candidate_names
    }


    for _, row in pairs.iterrows():
        candidate_a = row[
            "candidate_a"
        ]

        candidate_b = row[
            "candidate_b"
        ]

        connections[candidate_a].add(
            candidate_b
        )

        connections[candidate_b].add(
            candidate_a
        )


    group_by_candidate = {}

    visited = set()

    group_number = 0


    for candidate in candidate_names:
        if (
            candidate in visited
            or len(
                connections[candidate]
            ) == 0
        ):
            continue


        group_number += 1

        stack = [
            candidate
        ]


        while stack:
            current = stack.pop()

            if current in visited:
                continue


            visited.add(
                current
            )

            group_by_candidate[
                current
            ] = group_number


            for neighbor in connections[
                current
            ]:
                if neighbor not in visited:
                    stack.append(
                        neighbor
                    )


    color_cycle = [
        "C0",
        "C1",
        "C2",
        "C3",
        "C4",
        "C5",
        "C6",
        "C7",
        "C8",
        "C9",
    ]


    colors = []


    for candidate in candidate_names:
        if candidate in group_by_candidate:
            group = group_by_candidate[
                candidate
            ]

            colors.append(
                color_cycle[
                    (group - 1)
                    % len(
                        color_cycle
                    )
                ]
            )

        else:
            colors.append(
                "lightgray"
            )


    data[
        "case_group"
    ] = data[
        "canonical_candidate"
    ].map(
        group_by_candidate
    )

    return data, colors


# 8. LOOP THROUGH DISTRICTS 1–4 AND SAVE THE FIGURES

The notebook automatically saves one figure per district under:

`finance_vs_ballot_support/figures/finance_metric_vs_threshold_support/`


In [ ]:
if X_VARIABLE not in METRICS:
    raise ValueError(
        f"Unknown X_VARIABLE: {X_VARIABLE}"
    )


x_column = METRICS[
    X_VARIABLE
]["column"]

x_label = METRICS[
    X_VARIABLE
]["label"]


FIGURE_DIR = (
    ROOT
    / "finance_vs_ballot_support"
    / "figures"
    / "finance_metric_vs_threshold_support"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


models = {}
pair_tables = {}


for district in DISTRICTS:
    print()
    print("=" * 70)
    print(f"DISTRICT {district}")
    print("=" * 70)


    # -----------------------------------------------------------
    # Prepare this district
    # -----------------------------------------------------------

    plot_data = analysis[
        analysis["district"].eq(
            district
        )
    ].copy()


    plot_data = plot_data.dropna(
        subset=[
            x_column,
            Y_VARIABLE,
        ]
    ).copy()


    print("X variable:", x_label)
    print("Candidates in regression:", len(plot_data))


    # -----------------------------------------------------------
    # Find comparable candidates
    # -----------------------------------------------------------

    interesting_pairs = find_interesting_pairs(
        plot_data,
        x_column=x_column,
        y_column=Y_VARIABLE,
        x_tolerance=SIMILAR_X_TOLERANCE,
        y_gap_threshold=LARGE_Y_GAP,
    )


    print(
        "Interesting comparable pairs:",
        len(interesting_pairs),
    )


    pair_tables[
        district
    ] = interesting_pairs.copy()


    plot_data, point_colors = (
        assign_case_groups(
            plot_data,
            interesting_pairs,
        )
    )


    # -----------------------------------------------------------
    # Figure path
    # -----------------------------------------------------------

    figure_path = (
        FIGURE_DIR
        / (
            f"d{district}_"
            f"{X_VARIABLE}_"
            "vs_mentions_pct_threshold.png"
        )
    )


    # -----------------------------------------------------------
    # Plot
    # -----------------------------------------------------------

    title = (
        f"District {district}: "
        f"{x_label} vs "
        + (
            "ballot mentions"
            if Y_VARIABLE == "mentions"
            else "ballot mentions (% of STV threshold)"
        )
    )


    model = OLSlinearRegression(
        x_data=plot_data[
            x_column
        ],
        y_data=plot_data[
            Y_VARIABLE
        ],
        x_label=x_label,
        y_label="Ballot mentions (% of STV threshold)",
        title=title,
        save_name=(
            str(
                figure_path
            )
            if SAVE_FIGURES
            else ""
        ),
        colors=point_colors,
        names=plot_data[
            "canonical_candidate"
        ],
    )


    models[
        district
    ] = model


print()
print("Figures folder:")
print(FIGURE_DIR)


## 9. District regression summary

In [ ]:
summary_rows = []


for district, model in models.items():
    summary_rows.append(
        {
            "district": district,
            "n": int(model.nobs),
            "r_squared": model.rsquared,
            "slope": model.params[1],
            "slope_p_value": model.pvalues[1],
        }
    )


district_results = pd.DataFrame(
    summary_rows
)


display(
    district_results.round(
        {
            "r_squared": 3,
            "slope": 4,
            "slope_p_value": 4,
        }
    )
)
